> **Runtime: most cells run in JupyterLite; some need Google Colab.** — The embedding-loading section uses `gensim` + a small pretrained Word2Vec/GloVe model which is not available under JupyterLite. Sections 3 through 6 (visualisation, positional encoding, from-scratch attention) use only NumPy and run in the browser. Each section flags its runtime.
>
> [![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/L3GJ0N/course-notebooks-public/blob/main/03-embeddings-and-attention-playground.ipynb)
>
> **[Click here to open in Google Colab](https://colab.research.google.com/github/L3GJ0N/course-notebooks-public/blob/main/03-embeddings-and-attention-playground.ipynb)**

# Embeddings and Attention Playground

**Lecture 5 — Inside the Transformer: The Complete Forward Pass**

In this notebook you will turn the chapters of Lecture 5 into things you can run, tweak, and see. Four blocks:

1. **Embedding geometry** — cosine similarity, nearest neighbours, and the    classic `king − man + woman ≈ queen` experiment on real pretrained vectors.
2. **Latent-space visualisation** — project hundreds of embedding vectors down    to 2D and 3D with t-SNE / UMAP and watch semantic clusters emerge.
3. **Positional encoding — order matters** — implement attention on bare    embeddings, show that `shutil.copy(src, dst)` and `shutil.copy(dst, src)`    produce identical outputs up to permutation, then add a position signal and    watch the outputs diverge.
4. **From-scratch scaled dot-product attention** — 15 lines of NumPy that    reproduce the six-step worked example from Chapter 10 (the `cat` sentence    with 35% attention weight on the right word).

At the end of the notebook, write your findings in the provided answer cells.

## How to Run This Notebook

Most cells run directly in **JupyterLite** (your browser). The embedding-loading section (Section 2) needs `gensim` and a downloaded pretrained model — easiest way to get both is **Google Colab**; click the badge at the top.

If a section does not work in your runtime, skip to the next — each section is self-contained.

## 1. Setup

We'll need NumPy throughout, matplotlib for plots, and scikit-learn for t-SNE. UMAP is optional — if it is not installed we fall back to t-SNE alone.

**In JupyterLite**, `numpy`, `matplotlib`, and `scikit-learn` are pre-installed. **In Colab**, the `pip install` line below adds `umap-learn` and `gensim` for the later sections.

In [ ]:
# Works in both JupyterLite and Colab. In JupyterLite, pip-installs silently
# no-op for packages already bundled.
%pip install -q numpy matplotlib scikit-learn umap-learn 2>/dev/null || true

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # reproducible
print('NumPy', np.__version__)

## 2. Embedding geometry (Colab recommended)

> **Runtime note:** this section needs `gensim` and a downloaded pretrained model (~65 MB). Easiest in Colab. In JupyterLite, skip to Section 3.

We'll use the `glove-wiki-gigaword-50` vectors — 50-dimensional embeddings trained on Wikipedia + Gigaword by Pennington et al. (2014). Small enough to download in seconds, rich enough to show real semantic structure.

In [ ]:
# In Colab (and local Python), install gensim and download the model.
# In JupyterLite this cell will fail — that's expected, skip to Section 3.
try:
    %pip install -q gensim
    import gensim.downloader as api
    model = api.load('glove-wiki-gigaword-50')
    HAS_EMBEDDINGS = True
    print(f'Loaded {len(model)} word vectors, each of dimension {model.vector_size}.')
except Exception as e:
    HAS_EMBEDDINGS = False
    print(f'Skipping Section 2 ({type(e).__name__}: {e})')
    print('Run sections 3 onwards instead — they use only NumPy.')

### 2.1 Cosine similarity by hand

Recall from Chapter 4: cosine similarity strips length out of the dot product and measures pure direction. Let's implement it and try it on a few pairs.

In [ ]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

if HAS_EMBEDDINGS:
    pairs = [
        ('dog', 'puppy'),
        ('dog', 'cat'),
        ('dog', 'tuesday'),
        ('python', 'javascript'),
        ('python', 'snake'),
        ('python', 'breakfast'),
    ]
    for w1, w2 in pairs:
        sim = cosine_similarity(model[w1], model[w2])
        print(f'cos({w1:12s}, {w2:12s}) = {sim:+.3f}')

Notice how `dog`/`puppy` scores highest (semantic near-synonyms), `dog`/`cat` second (same category), and `dog`/`tuesday` near zero (unrelated). The vectors encode meaning as direction, and cosine similarity pulls it out.

### 2.2 Nearest neighbours

Given a word, find the closest other words in the embedding space. This is the classic smoke-test that embeddings encode semantic structure.

In [ ]:
if HAS_EMBEDDINGS:
    for word in ['python', 'king', 'paris', 'function']:
        neighbours = model.most_similar(word, topn=5)
        print(f'{word}:')
        for w, s in neighbours:
            print(f'    {w:15s} (sim={s:.3f})')
        print()

### 2.3 The `king − man + woman ≈ queen` experiment

From Chapter 4. We subtract the vector for `man`, add the vector for `woman`, and look for the closest word to the result (excluding the three input words). If meaning lives in direction, the direction we travel should be the 'masculine → feminine' axis, and applying it to `king` should land us near `queen`.

**Honest disclaimer**: this experiment is partly an artefact of how `gensim` excludes the input words from the search. See [Nissim et al. 2019](https://arxiv.org/abs/1905.09866) for a thorough critique. Still, the underlying principle — directions in embedding space carry meaning — is real.

In [ ]:
if HAS_EMBEDDINGS:
    for a, b, c in [
        ('king', 'man', 'woman'),
        ('paris', 'france', 'germany'),
        ('walked', 'walking', 'running'),
        ('bigger', 'big', 'small'),
    ]:
        # compute model[a] - model[b] + model[c] and find the closest word
        results = model.most_similar(positive=[a, c], negative=[b], topn=3)
        print(f'{a} - {b} + {c} ≈')
        for w, s in results:
            print(f'    {w:15s} (sim={s:.3f})')
        print()

**Questions to answer:**

- Did `king − man + woman` land near `queen`? How confident is the match?
- Try your own analogies — one that works and one that does not. Report both.
- What happens when you try it with less common words? Why might the effect weaken?

## 3. Latent-space visualisation

We cannot plot 50-dimensional vectors directly. But t-SNE and UMAP let us project them down to 2D or 3D while trying to preserve local neighbourhood structure. The goal is to *see* clusters of semantically related tokens.

This section runs in JupyterLite if you have the model loaded. Otherwise it uses a bundled toy corpus defined below.

In [ ]:
# A hand-picked vocabulary for visualisation. Mix of semantic categories so
# we can colour-code and check whether the projection groups them correctly.
VOCAB = {
    'code': ['def', 'return', 'if', 'else', 'for', 'while', 'import', 'class',
             'print', 'true', 'false', 'none'],
    'animals': ['dog', 'cat', 'horse', 'cow', 'rabbit', 'mouse', 'bird', 'fish',
                'lion', 'tiger', 'bear', 'fox'],
    'numbers': ['one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight',
                'nine', 'ten', 'hundred', 'thousand'],
    'countries': ['germany', 'france', 'spain', 'italy', 'china', 'japan',
                  'brazil', 'canada', 'russia', 'india', 'australia', 'egypt'],
    'cities': ['berlin', 'paris', 'madrid', 'rome', 'beijing', 'tokyo',
               'moscow', 'london', 'sydney', 'cairo', 'delhi', 'ottawa'],
    'colors': ['red', 'blue', 'green', 'yellow', 'black', 'white', 'purple',
               'orange', 'pink', 'brown', 'gray', 'violet'],
}

all_words = [w for cat in VOCAB.values() for w in cat]
category_of = {w: cat for cat, words in VOCAB.items() for w in words}
print(f'{len(all_words)} words across {len(VOCAB)} categories.')

Build a matrix of embedding vectors for these words. If real embeddings are available we use them. Otherwise we fabricate *structured random* vectors that cluster by category — enough for students to see the projection algorithm work, even if the demo is cosmetic.

In [ ]:
if HAS_EMBEDDINGS:
    # Real embeddings — skip words the model does not know
    known_words = [w for w in all_words if w in model]
    X = np.array([model[w] for w in known_words])
    labels = [category_of[w] for w in known_words]
    print(f'Using real GloVe embeddings for {len(known_words)} known words.')
else:
    # Fallback: fabricate 50-d vectors that cluster by category
    rng = np.random.default_rng(42)
    known_words = all_words
    labels = [category_of[w] for w in known_words]
    # Per-category centre, plus noise
    centres = {cat: rng.normal(size=50) * 3 for cat in VOCAB}
    X = np.array([centres[labels[i]] + rng.normal(size=50) for i in range(len(known_words))])
    print(f'Using fabricated clustered vectors for {len(known_words)} words '
          '(JupyterLite fallback — real embeddings unavailable).')

print(f'Embedding matrix shape: {X.shape}')

### 3.1 t-SNE in 2D

t-SNE optimises for preserving local neighbourhoods. Similar words should sit near each other in the projection.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=10, random_state=42, init='pca')
X_2d = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(10, 8))
cats = list(VOCAB.keys())
cmap = plt.get_cmap('tab10')
for i, cat in enumerate(cats):
    mask = [labels[j] == cat for j in range(len(labels))]
    pts = X_2d[mask]
    ax.scatter(pts[:, 0], pts[:, 1], c=[cmap(i)], label=cat, s=50)
    for j in range(len(labels)):
        if labels[j] == cat:
            ax.annotate(known_words[j], (X_2d[j, 0], X_2d[j, 1]),
                        fontsize=8, alpha=0.7)
ax.legend(loc='best', fontsize=9)
ax.set_title('t-SNE projection of word embeddings')
ax.set_xlabel('t-SNE component 1')
ax.set_ylabel('t-SNE component 2')
plt.tight_layout()
plt.show()

### 3.2 UMAP in 3D (if available)

UMAP tries to preserve both local and global structure. A 3D projection can be rotated to reveal clusters that overlap in 2D. We'll fall back to a 2D plot if UMAP is not installed.

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_components=3, n_neighbors=10, min_dist=0.3, random_state=42)
    X_3d = reducer.fit_transform(X)
    HAS_UMAP = True
except Exception as e:
    HAS_UMAP = False
    print(f'UMAP unavailable ({type(e).__name__}: {e}); falling back to 3D t-SNE.')
    reducer = TSNE(n_components=3, perplexity=10, random_state=42, init='pca')
    X_3d = reducer.fit_transform(X)

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3d projection)
fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection='3d')
for i, cat in enumerate(cats):
    mask = [labels[j] == cat for j in range(len(labels))]
    pts = X_3d[np.array(mask)]
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=[cmap(i)], label=cat, s=50)
ax.set_title(f'{"UMAP" if HAS_UMAP else "t-SNE"} 3D projection')
ax.legend()
plt.tight_layout()
plt.show()

**Questions to answer:**

- Do countries and cities form distinct clusters, or do they overlap? Why?
- In the 3D view, are any categories especially close to each other? Does the   proximity match your intuition?
- If you re-run t-SNE with a different `perplexity` (try 5 or 30), how do the   clusters change? What does that tell you about trusting t-SNE distances?

## 4. Positional encoding — order matters

This is the demo from Chapter 5: attention on bare embeddings is permutation-equivariant, which means it cannot distinguish `shutil.copy(src, dst)` from `shutil.copy(dst, src)`. Let's verify it in code.

We'll use fabricated 4-dimensional embeddings for six tokens representing the call. The specific numbers do not matter — what matters is that swapping two tokens in the input produces the same outputs in a different order.

In [ ]:
TOKENS = ['shutil', '.', 'copy', '(', 'src', 'dst', ')']
# Fabricated 4-d embeddings for each token, fixed for reproducibility
rng = np.random.default_rng(0)
E = {tok: rng.normal(size=4) for tok in TOKENS}

def sequence_to_X(tokens):
    """Look up each token's embedding and stack into a T×d matrix."""
    return np.array([E[t] for t in tokens])

# Two 'sentences' that differ only in the order of 'src' and 'dst'
seq_correct = ['shutil', '.', 'copy', '(', 'src', 'dst', ')']
seq_swapped = ['shutil', '.', 'copy', '(', 'dst', 'src', ')']

X1 = sequence_to_X(seq_correct)
X2 = sequence_to_X(seq_swapped)

print(f'X1 shape: {X1.shape} — rows in order: {seq_correct}')
print(f'X2 shape: {X2.shape} — rows in order: {seq_swapped}')

### 4.1 Scaled dot-product attention (no position)

Implement attention in a handful of lines. Fixed projections `W_Q, W_K, W_V` — the same for both sequences.

In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def attention(X, WQ, WK, WV):
    """Single-head scaled dot-product attention."""
    Q = X @ WQ
    K = X @ WK
    V = X @ WV
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

# Fix the three projection matrices (small, same for both runs)
rng2 = np.random.default_rng(1)
WQ = rng2.normal(size=(4, 4))
WK = rng2.normal(size=(4, 4))
WV = rng2.normal(size=(4, 4))

Y1, A1 = attention(X1, WQ, WK, WV)
Y2, A2 = attention(X2, WQ, WK, WV)

print('Output rows of Y1 (correct order) indexed by their input token:')
for t, row in zip(seq_correct, Y1):
    print(f'  {t:7s}: {row.round(3)}')
print()
print('Output rows of Y2 (swapped order) indexed by their input token:')
for t, row in zip(seq_swapped, Y2):
    print(f'  {t:7s}: {row.round(3)}')

### 4.2 Verify the equivariance

If attention is permutation-equivariant, the *output row for `src`* in the correct sequence must match the *output row for `src`* in the swapped sequence — even though `src` sits at a different position. Same for every token.

In [ ]:
# Align the outputs by their input token (not by row index) and compare.
# Build dicts: token -> output row
map1 = {t: row for t, row in zip(seq_correct, Y1)}
map2 = {t: row for t, row in zip(seq_swapped, Y2)}

print(f'{"Token":10s}  {"|Y1[tok] - Y2[tok]|":20s}  {"Equal?":10s}')
print('-' * 50)
for t in seq_correct:
    diff = np.linalg.norm(map1[t] - map2[t])
    ok = 'yes' if diff < 1e-9 else 'NO'
    print(f'{t:10s}  {diff:.2e}             {ok}')

Every row matches to floating-point precision. The model saw two different input sequences, but produced outputs whose *content*, per token, is indistinguishable. In a real model this means `shutil.copy(src, dst)` and `shutil.copy(dst, src)` are treated as the same computation — which is exactly the bug we set out to rule out.

### 4.3 Add a positional signal

Now add a sinusoidal positional encoding to each row before attention sees it, and watch the outputs diverge.

In [ ]:
def sinusoidal_position(T, d):
    """Classic Vaswani-et-al sinusoidal positional encoding, T×d."""
    P = np.zeros((T, d))
    pos = np.arange(T)[:, None]
    i = np.arange(d)[None, :]
    denom = 10000 ** (2 * (i // 2) / d)
    P[:, 0::2] = np.sin(pos / denom[:, 0::2])
    P[:, 1::2] = np.cos(pos / denom[:, 1::2])
    return P

T, d = X1.shape
P = sinusoidal_position(T, d)

X1_pos = X1 + P
X2_pos = X2 + P  # same P — because positions 0..6 are the same, only the token at each position differs

Y1_pos, _ = attention(X1_pos, WQ, WK, WV)
Y2_pos, _ = attention(X2_pos, WQ, WK, WV)

map1p = {t: row for t, row in zip(seq_correct, Y1_pos)}
map2p = {t: row for t, row in zip(seq_swapped, Y2_pos)}

print('Per-token output diff WITH positional encoding:')
print(f'{"Token":10s}  {"|Y1[tok] - Y2[tok]|":20s}  {"Now distinguishable?":20s}')
print('-' * 60)
for t in seq_correct:
    diff = np.linalg.norm(map1p[t] - map2p[t])
    ok = 'yes' if diff > 1e-3 else 'no (still equivariant)'
    print(f'{t:10s}  {diff:.4f}              {ok}')

With position added, the output for `src` in the correct sequence is genuinely different from the output for `src` in the swapped sequence. The model can now tell them apart. This is why positional encoding is not optional.

**Questions to answer:**

- Before position is added, the per-token output diffs were effectively zero.   After position is added, how big did they get on your run?
- Try swapping two other tokens in the sequence (e.g. `.` and `copy`). Does   the effect hold?
- What if you multiply the positional signal by a small constant (say 0.01)   before adding — does the model still distinguish the orders? What does that   tell you about tuning the relative scale of token vs. positional signal?

## 5. From-scratch scaled dot-product attention

The six-step worked example from Chapter 10, in code. The sentence is *"The cat sat on it"* and we're computing the attention output for the token `it`. At the end you should see 35% of the attention weight landing on `cat` — exactly the number from the lecture.

We'll hardcode the query and keys so the arithmetic matches the lecture.

In [ ]:
# From Chapter 10, exact numbers
TOKENS = ['The', 'cat', 'sat', 'on', 'it']
q_it = np.array([1.0, 0.0, 1.0])
K = np.array([
    [0.5, 0.5, 0.0],  # The
    [1.0, 0.5, 1.5],  # cat
    [1.0, 0.5, 0.5],  # sat
    [0.5, 0.5, 1.0],  # on
    [0.0, 1.0, 1.0],  # it
])
# Fabricated values (any numbers work — these match the lecture illustration)
V = np.array([
    [0.2, 0.1, 0.3],
    [1.0, 0.8, 1.2],
    [0.8, 0.6, 0.9],
    [0.5, 0.7, 0.6],
    [0.9, 1.0, 0.8],
])
d_k = 3

# --- Step 2: Score ---
scores = q_it @ K.T
print(f'Step 2 — scores (q_it · k_t):')
for t, s in zip(TOKENS, scores):
    print(f'  {t:5s}: {s:.2f}')

# --- Step 3: Scale ---
scaled = scores / np.sqrt(d_k)
print(f'\nStep 3 — scaled (÷√{d_k}≈{np.sqrt(d_k):.2f}):')
for t, s in zip(TOKENS, scaled):
    print(f'  {t:5s}: {s:.2f}')

# --- Step 4: Softmax ---
weights = np.exp(scaled) / np.exp(scaled).sum()
print(f'\nStep 4 — softmax (attention weights):')
for t, w in zip(TOKENS, weights):
    bar = '█' * int(w * 50)
    print(f'  {t:5s}: {w:.2f}  {bar}')

# --- Step 5+6: Weight and Sum ---
context = weights @ V
print(f'\nSteps 5+6 — weighted sum of values:')
print(f'  context_it = {context.round(3)}')

### 5.1 Try changing the numbers

Your turn. Pick an entry in the keys `K` and push it up or down. See how the attention weights redistribute. Can you make `it` attend most strongly to `sat` instead of `cat`?

In [ ]:
# TODO: edit K (or q_it) and rerun. Watch the attention weights shift.
# For example, make 'sat' the winner by pushing its key to be more aligned with q_it:

K_edit = K.copy()
K_edit[2] = np.array([1.5, 0.0, 1.5])  # 'sat' now aligns strongly with q_it=[1,0,1]

scores2 = q_it @ K_edit.T
scaled2 = scores2 / np.sqrt(d_k)
weights2 = np.exp(scaled2) / np.exp(scaled2).sum()

print('After editing k_sat:')
for t, w in zip(TOKENS, weights2):
    bar = '█' * int(w * 50)
    print(f'  {t:5s}: {w:.2f}  {bar}')

### 5.2 Scaling sanity-check

From Chapter 11: without the `√d_k` division, softmax saturates as `d_k` grows. Let's simulate that.

In [ ]:
for d_test in [3, 16, 64, 256]:
    rng3 = np.random.default_rng(7)
    q = rng3.normal(size=d_test)
    keys = rng3.normal(size=(5, d_test))
    raw = q @ keys.T
    unscaled_probs = np.exp(raw) / np.exp(raw).sum()
    scaled_probs = np.exp(raw / np.sqrt(d_test)) / np.exp(raw / np.sqrt(d_test)).sum()
    print(f'd_k = {d_test:3d}')
    print(f'  unscaled softmax max prob: {unscaled_probs.max():.4f} (1.0 = fully saturated)')
    print(f'  scaled   softmax max prob: {scaled_probs.max():.4f}')
    print()

As `d_k` grows, the unscaled softmax collapses onto a single winner. The scaled version stays spread out. That is the whole story of Chapter 11 in ten lines of code.

## 6. Your findings

Answer the prompts below in the provided markdown cells. Double-click to edit. If you are in JupyterLite, your answers save in your browser. If you are in Colab, use **File → Save a copy** or **File → Download** to keep them.

### 1. The geometry recap

Pick two analogies you ran in Section 2.3. For each, what was the top prediction and how confident was it? If one worked and one did not, speculate about why based on what you saw in the nearest-neighbours experiment (Section 2.2).

*Your answer here*

### 2. Latent-space clusters

From the 2D and 3D projection in Section 3: which categories came out cleanly separated? Which ones overlapped or were harder to tell apart? Are the overlapping pairs *semantically* related in ways that might explain it?

*Your answer here*

### 3. Position matters

Paste the output of Section 4.2 and Section 4.3 (the per-token diffs, without and with positional encoding). In one sentence, what do the numbers say? Why does this experiment rule out the naive 'just apply attention to embeddings' design?

*Your answer here*

### 4. From-scratch attention

In Section 5.1 you edited the keys. Describe one edit you made and the effect it had on the attention distribution. If this layer were learning during training, what would gradient descent be trying to adjust about these keys?

*Your answer here*

### 5. One thing that changes how you prompt

Based on everything you now know — embeddings as directions, positional encodings as the reason order matters, attention as a finite budget being divided across all tokens — what is one concrete change you will make to how you write prompts or CLAUDE.md files going forward?

*Your answer here*